In [7]:
import pandas as pd
from tabulate import tabulate

# Load anomaly detection results from each model
df_autoencoder = pd.read_csv("Software_Anomalies_Autoencoder_Decoded.csv")
df_if = pd.read_csv("Software_Anomalies_IF.csv")
df_svm = pd.read_csv("Software_Anomalies_SVM.csv")
df_lof = pd.read_csv("Software_Anomalies_LOF.csv")

# Assign flag (1 = detected as anomaly)
df_autoencoder["Autoencoder"] = 1
df_if["IF"] = 1
df_svm["SVM"] = 1
df_lof["LOF"] = 1

# Merge all models based on Product_Name
df_combined = (
    df_autoencoder[["Product_Name", "Autoencoder"]]
    .merge(df_if[["Product_Name", "IF"]], on="Product_Name", how="outer")
    .merge(df_svm[["Product_Name", "SVM"]], on="Product_Name", how="outer")
    .merge(df_lof[["Product_Name", "LOF"]], on="Product_Name", how="outer")
    .fillna(0)  # Fill missing values with 0 (not flagged)
)

# Convert to integer (since NaN was replaced with 0)
df_combined[["Autoencoder", "IF", "SVM", "LOF"]] = df_combined[["Autoencoder", "IF", "SVM", "LOF"]].astype(int)

# Count how many models flagged each software
df_combined["Total_Flags"] = df_combined[["Autoencoder", "IF", "SVM", "LOF"]].sum(axis=1)

# Filter software flagged by at least 3 models
df_flagged_3plus = df_combined[df_combined["Total_Flags"] >= 3]

# Drop duplicate software names (if they exist)
df_unique_flagged = df_flagged_3plus.drop_duplicates(subset=["Product_Name"])

# Save results
df_unique_flagged.to_csv("Unique_Software_Flagged_By_At_Least_3_Models.csv", index=False)

# Print summary with neat table format
print(f"✅ Found {len(df_unique_flagged)} UNIQUE software flagged by at least 3 models!\n")

# Display only the first 20 rows for readability
print(tabulate(df_unique_flagged.head(20), headers="keys", tablefmt="pretty"))


✅ Found 3 UNIQUE software flagged by at least 3 models!

+------+------------------------------------------------+-------------+----+-----+-----+-------------+
|      |                  Product_Name                  | Autoencoder | IF | SVM | LOF | Total_Flags |
+------+------------------------------------------------+-------------+----+-----+-----+-------------+
|  68  |    kaspersky endpoint security for windows     |      0      | 1  |  1  |  1  |      3      |
| 3929 | microsoft office access runtime (english) 2007 |      0      | 1  |  1  |  1  |      3      |
| 3956 |       microsoft visio professional 2013        |      0      | 1  |  1  |  1  |      3      |
+------+------------------------------------------------+-------------+----+-----+-----+-------------+
